# Amazon ML Challenge 2026 — Business Entity Resolution
## High-Throughput TPU Model Training & Inference Pipeline

This notebook trains a **Deep Residual Entity Matcher (PyTorch / PyTorch-XLA)** on Google Colab **TPU** (Tensor Processing Unit).

### How to run with the VS Code Google Colab Extension:
1. In VS Code, click **Select Kernel** at top-right of this notebook.
2. Select **Google Colab** -> Sign in with your Google account.
3. In runtime options, select **TPU** (e.g. TPU v2/v3 or v5e).
4. Run the cells below! Cells will execute directly on Colab's cloud TPU.


In [ ]:
# 1. Environment & TPU Hardware Detection
import os
import sys
import time

print("=" * 60)
print("HARDWARE ACCELERATOR DIAGNOSTICS")
print("=" * 60)

try:
    import torch
    import torch_xla
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    DEVICE_TYPE = "tpu"
    print(f"[+] TPU Initialized Successfully via PyTorch XLA!")
    print(f"[+] TPU Device: {device}")
    print(f"[+] PyTorch: {torch.__version__} | Torch XLA: {torch_xla.__version__}")
except Exception as e:
    import torch
    if torch.cuda.is_available():
        device = torch.device("cuda")
        DEVICE_TYPE = "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        DEVICE_TYPE = "mps"
    else:
        device = torch.device("cpu")
        DEVICE_TYPE = "cpu"
    print(f"[!] Running on {DEVICE_TYPE.upper()} device: {device} ({e})")


HARDWARE ACCELERATOR DIAGNOSTICS


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install High-Performance Dependencies
!pip install -q rapidfuzz polars lightgbm scikit-learn


In [ ]:
# 3. Mount Google Drive or Prepare Workspace
# If running on Colab Cloud, you can mount Google Drive to access or store datasets:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("[+] Google Drive mounted at /content/drive")
except Exception:
    print("[*] Running locally or Drive already mounted.")

# Verify dataset directories exist
os.makedirs("model", exist_ok=True)
os.makedirs("output", exist_ok=True)

train_exists = os.path.exists("dataset/train/train_source1.tsv")
test_exists = os.path.exists("dataset/test/test_source1.tsv")
print(f"Training files found: {train_exists}")
print(f"Test files found: {test_exists}")


In [ ]:
# 4. Define Deep Residual Entity Matcher Architecture for TPU
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout=0.15):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.act1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.act2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.drop2(self.act2(self.norm2(self.fc2(self.drop1(self.act1(self.norm1(self.fc1(x))))))))

class DeepEntityMatcher(nn.Module):
    def __init__(self, in_features=15, hidden_dim=128, num_blocks=3, dropout=0.15):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        h = self.input_layer(x)
        for block in self.blocks:
            h = block(h)
        return self.head(h).squeeze(-1)

print("[+] DeepEntityMatcher model architecture ready!")


In [ ]:
# 5. Train Model on TPU
from src.train_tpu import train_model

model, best_threshold = train_model(
    train_dir="dataset/train",
    model_dir="model",
    top_k=25,
    val_fraction=0.15,
    max_train_entities=50000, # Adjust based on desired training speed
    epochs=8,
    batch_size=2048,           # Large batch size maximizes TPU MXU throughput
    lr=1e-3,
    seed=42,
)

print(f"
[+] Training complete! Best validation threshold: {best_threshold:.2f}")


In [ ]:
# 6. Run Fast Test Inference to Generate Leaderboard Outputs
from src.predict import run_prediction

run_prediction(
    test_dir="dataset/test",
    output_dir="output",
    max_cands_per_s1=30,
)


In [ ]:
# 7. Validate Submissions with Official Competition Validator
!python student_resource/utils/validate_submission.py \n
    --matching output/matching_results.tsv \n
    --candidate output/candidate_pairs.tsv \n
    --test-dir dataset/test
